In [5]:
# Check available variables in 2022 full dataset
!pip install pyreadstat -q

from google.colab import drive
drive.mount('/content/drive')

import pyreadstat
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

sav_base = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/data/raw/.SAV files/'

df_test, meta = pyreadstat.read_sav(
    sav_base + '2022CY08MSP_STUDENT_QQQ.SAV',
    usecols=['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T',
             'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV1READ', 'PV1SCIE'],
    row_limit=5
)

print("Available columns:")
print(list(df_test.columns))
print(f"\nSample rows:")
print(df_test.head())
print(f"\nTotal rows in dataset: {meta.number_rows:,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries loaded successfully
Available columns:
['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T', 'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV1READ', 'PV1SCIE']

Sample rows:
   CNT  CNTSCHID  CNTSTUID  ST004D01T    ESCS  W_FSTUWT  PV1MATH  PV1READ  \
0  ALB  800282.0  800001.0        1.0  1.1112   3.15874  179.583  247.571   
1  ALB  800115.0  800002.0        2.0 -3.0507   4.34524  308.247  258.472   
2  ALB  800242.0  800003.0        2.0 -0.1867   7.83860  267.514  284.670   
3  ALB  800245.0  800005.0        1.0 -3.2198   8.49148  272.649  321.547   
4  ALB  800285.0  800006.0        1.0 -1.0548   3.70957  435.473  464.366   

   PV1SCIE  
0  335.468  
1  315.021  
2  358.675  
3  214.823  
4  434.997  

Total rows in dataset: 5


In [4]:
raw_path = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/data/raw/'
for f in os.listdir(raw_path):
    print(repr(f))

'.SAV files'


In [6]:
# Check full dataset size and ESCS distribution
df_2022_full, meta_2022 = pyreadstat.read_sav(
    sav_base + '2022CY08MSP_STUDENT_QQQ.SAV',
    usecols=['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T',
             'ESCS', 'W_FSTUWT', 'PV1MATH', 'PV1READ', 'PV1SCIE']
)

print(f"Total rows: {len(df_2022_full):,}")
print(f"Countries: {df_2022_full['CNT'].nunique()}")
print(f"\nStudents per country (sample):")
print(df_2022_full.groupby('CNT').size().describe().round(0).to_string())
print(f"\nESCS range: {df_2022_full['ESCS'].min():.2f} to {df_2022_full['ESCS'].max():.2f}")
print(f"ESCS missing: {df_2022_full['ESCS'].isna().sum():,}")

Total rows: 613,744
Countries: 80

Students per country (sample):
count       80.0
mean      7672.0
std       4453.0
min       3127.0
25%       5985.0
50%       6536.0
75%       7311.0
max      30800.0

ESCS range: -6.84 to 7.38
ESCS missing: 25,468


In [9]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
!pip install pyreadstat -q

from google.colab import drive
drive.mount('/content/drive')

import pyreadstat
import pandas as pd
import numpy as np
import re
import os

base = '/content/drive/MyDrive/Analytics for Society/Analytics for Society Award/'
sav_base = base + 'data/raw/.SAV files/'
processed_base = base + 'data/processed/'

print("Setup complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete


In [35]:
# ── CELL 2: Stratified sampling function ──────────────────────────────────────

def stratified_sample_pisa(df, n_total=2000, random_state=42):
    """
    Two-variable stratified sample per country:
    - Primary stratum:   ESCS quartile (Q1-Q4)
    - Secondary stratum: school type (public/private)

    Approximates PISA's two-stage design:
    Stage 1 — school selection stratified by type
    Stage 2 — student selection within schools

    Target: n_total students per country
    """
    results = []
    skipped = []

    for cnt, group in df.groupby('CNT'):
        group_clean = group.dropna(subset=['ESCS']).copy()

        if len(group_clean) < 50:
            skipped.append(f"{cnt} (too few students)")
            continue

        # Assign within-country ESCS quartile
        try:
            group_clean['ESCS_Q'] = pd.qcut(
                group_clean['ESCS'],
                q=4,
                labels=['Q1_low', 'Q2_mid_low', 'Q3_mid_high', 'Q4_high'],
                duplicates='drop'
            )
        except ValueError:
            skipped.append(f"{cnt} (ESCS quartile error)")
            continue

        # Check if school type available
        has_schtype = ('SCHTYPE' in group_clean.columns and
                       group_clean['SCHTYPE'].notna().sum() > 0)

        if has_schtype:
            # Two-variable stratification: ESCS quartile × school type
            strata = group_clean.groupby(
                ['ESCS_Q', 'SCHTYPE'], observed=True
            )
            n_strata = len(strata)
            n_per_stratum = max(3, n_total // n_strata)

            samples = []
            for _, stratum_group in strata:
                n_s = min(len(stratum_group), n_per_stratum)
                samples.append(
                    stratum_group.sample(n_s, random_state=random_state)
                )
            sampled = pd.concat(samples, ignore_index=True)

            # Top up to n_total if under target
            if len(sampled) < n_total:
                remaining = group_clean[
                    ~group_clean.index.isin(sampled.index)
                ]
                top_up = min(n_total - len(sampled), len(remaining))
                if top_up > 0:
                    sampled = pd.concat([
                        sampled,
                        remaining.sample(top_up, random_state=random_state)
                    ], ignore_index=True)

        else:
            # Fallback: ESCS-only stratification
            n_per_q = n_total // 4
            samples = []
            for q in ['Q1_low', 'Q2_mid_low', 'Q3_mid_high', 'Q4_high']:
                q_group = group_clean[group_clean['ESCS_Q'] == q]
                if len(q_group) > 0:
                    samples.append(
                        q_group.sample(
                            min(len(q_group), n_per_q),
                            random_state=random_state
                        )
                    )
            sampled = pd.concat(samples, ignore_index=True)

        # Drop helper columns
        sampled = sampled.drop(
            columns=[c for c in ['ESCS_Q'] if c in sampled.columns]
        )
        results.append(sampled)

    if skipped:
        print(f"  Skipped {len(skipped)} countries: {skipped[:5]}")

    return pd.concat(results, ignore_index=True)

print("Stratified sampling function defined")

Stratified sampling function defined


In [26]:
# ── CELL 3: Helper functions ───────────────────────────────────────────────────

# ── CELL 3: Helper functions ───────────────────────────────────────────────────

def parse_spss_syntax(syntax_file):
    with open(syntax_file, 'r', encoding='latin-1') as f:
        content = f.read()
    pattern = r'(\w+)\s+(\d+)\s*-\s*(\d+)'
    matches = re.findall(pattern, content)
    col_names, col_specs, seen = [], [], {}
    for m in matches:
        name = m[0]
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        col_names.append(name)
        col_specs.append((int(m[1])-1, int(m[2])))
    return col_names, col_specs


def load_school_schtype(year, sav_base):
    school_files = {
        '2009': ('2009INT_SCHOOL09_Dec11.txt',
                 'PISA2009_SPSS_school.txt', 'txt'),
        '2012': ('2012INT_SCHOOL12_DEC03.txt',
                 'SPSS 2012 syntax to read in school questionnaire data file.txt', 'txt'),
        '2015': ('2015CY6_MS_CMB_SCH_QQQ.sav', None, 'sav'),
        '2018': ('2018CY07_MSU_SCH_QQQ.sav',   None, 'sav'),
        '2022': ('2022CY08MSP_SCHOOL_QQQ.SAV',  None, 'sav')
    }

    fname, syntax, ftype = school_files[year]

    if ftype == 'sav':
        df_sch, _ = pyreadstat.read_sav(
            sav_base + fname,
            usecols=['CNT', 'CNTSCHID', 'SCHLTYPE']
        )
        df_sch = df_sch.rename(columns={
            'CNTSCHID': 'SCHOOLID',
            'SCHLTYPE': 'SCHTYPE'
        })

    else:
        col_names, col_specs = parse_spss_syntax(sav_base + syntax)

        schtype_variants = ['SCHTYPE', 'SCHLTYPE']
        schtype_col = next(
            (c for c in col_names if c in schtype_variants), None
        )

        if schtype_col is None:
            print(f"  WARNING: No school type column found for {year}")
            return pd.DataFrame(columns=['CNT', 'SCHOOLID', 'SCHTYPE'])

        needed = ['CNT', 'SCHOOLID', schtype_col]
        av_names, av_specs = [], []
        for name, spec in zip(col_names, col_specs):
            if name in needed:
                av_names.append(name)
                av_specs.append(spec)

        print(f"  Columns found in {year} school file: {av_names}")

        chunks = []
        for i, chunk in enumerate(pd.read_fwf(
            sav_base + fname,
            colspecs=av_specs,
            names=av_names,
            encoding='latin-1',
            chunksize=10000
        )):
            chunks.append(chunk)
            if i % 10 == 0:
                print(f"  School file: processed {i*10000:,} rows...")

        df_sch = pd.concat(chunks, ignore_index=True)

        if schtype_col == 'SCHLTYPE':
            df_sch = df_sch.rename(columns={'SCHLTYPE': 'SCHTYPE'})

    df_sch['SCHOOLID'] = pd.to_numeric(df_sch['SCHOOLID'], errors='coerce')
    df_sch['SCHTYPE']  = pd.to_numeric(df_sch['SCHTYPE'],  errors='coerce')
    df_sch = df_sch[['CNT', 'SCHOOLID', 'SCHTYPE']].drop_duplicates()
    print(f"  School types loaded for {year}: {len(df_sch):,} schools")
    return df_sch


print("Helper functions defined")

Helper functions defined


In [36]:
# ── CELL 4: Process SAV years (2015, 2018, 2022) ──────────────────────────────

sav_student_configs = {
    '2015': {
        'file': '2015CY6_MS_CMB_STU_QQQ.sav',
        'cols': ['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T',
                 'ESCS', 'W_FSTUWT',
                 'PV1MATH','PV2MATH','PV3MATH','PV4MATH','PV5MATH',
                 'PV6MATH','PV7MATH','PV8MATH','PV9MATH','PV10MATH',
                 'PV1READ','PV2READ','PV3READ','PV4READ','PV5READ',
                 'PV6READ','PV7READ','PV8READ','PV9READ','PV10READ',
                 'PV1SCIE','PV2SCIE','PV3SCIE','PV4SCIE','PV5SCIE',
                 'PV6SCIE','PV7SCIE','PV8SCIE','PV9SCIE','PV10SCIE'],
    },
    '2018': {
        'file': '2018CY07_MSU_STUDENT_QQQ.sav',
        'cols': ['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T',
                 'ESCS', 'W_FSTUWT',
                 'PV1MATH','PV2MATH','PV3MATH','PV4MATH','PV5MATH',
                 'PV6MATH','PV7MATH','PV8MATH','PV9MATH','PV10MATH',
                 'PV1READ','PV2READ','PV3READ','PV4READ','PV5READ',
                 'PV6READ','PV7READ','PV8READ','PV9READ','PV10READ',
                 'PV1SCIE','PV2SCIE','PV3SCIE','PV4SCIE','PV5SCIE',
                 'PV6SCIE','PV7SCIE','PV8SCIE','PV9SCIE','PV10SCIE'],
    },
    '2022': {
        'file': '2022CY08MSP_STUDENT_QQQ.SAV',
        'cols': ['CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T',
                 'ESCS', 'W_FSTUWT',
                 'PV1MATH','PV2MATH','PV3MATH','PV4MATH','PV5MATH',
                 'PV6MATH','PV7MATH','PV8MATH','PV9MATH','PV10MATH',
                 'PV1READ','PV2READ','PV3READ','PV4READ','PV5READ',
                 'PV6READ','PV7READ','PV8READ','PV9READ','PV10READ',
                 'PV1SCIE','PV2SCIE','PV3SCIE','PV4SCIE','PV5SCIE',
                 'PV6SCIE','PV7SCIE','PV8SCIE','PV9SCIE','PV10SCIE'],
    }
}

all_years = []

for year, config in sav_student_configs.items():
    print(f"\nProcessing {year}...")

    # Load students
    df_stu, _ = pyreadstat.read_sav(
        sav_base + config['file'],
        usecols=config['cols']
    )
    print(f"  Students loaded: {len(df_stu):,} rows")

    # Average plausible values
    df_stu['MATH']    = df_stu[[f'PV{i}MATH' for i in range(1,11)]].mean(axis=1)
    df_stu['READ']    = df_stu[[f'PV{i}READ' for i in range(1,11)]].mean(axis=1)
    df_stu['SCIENCE'] = df_stu[[f'PV{i}SCIE' for i in range(1,11)]].mean(axis=1)
    df_stu = df_stu.drop(
        columns=[c for c in df_stu.columns if c.startswith('PV')]
    )

    # Rename student columns
    df_stu = df_stu.rename(columns={
        'CNTSCHID':  'SCHOOLID',
        'CNTSTUID':  'STUDENTID',
        'ST004D01T': 'GENDER',
        'W_FSTUWT':  'STU_WEIGHT'
    })

    # Load and join school type
    df_sch = load_school_schtype(year, sav_base)
    df_stu = df_stu.merge(df_sch, on=['CNT', 'SCHOOLID'], how='left')
    matched = df_stu['SCHTYPE'].notna().sum()
    print(f"  School type matched: {matched:,} / {len(df_stu):,} students")

    # Stratified sample
    df_sampled = stratified_sample_pisa(df_stu, n_total=2000)
    df_sampled['YEAR'] = year
    print(f"  Sampled: {len(df_sampled):,} rows, "
          f"{df_sampled['CNT'].nunique()} countries")

    df_sampled.to_csv(
        processed_base + f'pisa_student_{year}_stratified.csv',
        index=False
    )
    print(f"  Saved: pisa_student_{year}_stratified.csv")
    all_years.append(df_sampled)

print("\nSAV years complete")


Processing 2015...
  Students loaded: 519,334 rows
  School types loaded for 2015: 17,908 schools
  School type matched: 451,337 / 519,334 students
  Skipped 1 countries: ['ALB (too few students)']
  Sampled: 142,878 rows, 72 countries
  Saved: pisa_student_2015_stratified.csv

Processing 2018...
  Students loaded: 612,004 rows
  School types loaded for 2018: 21,903 schools
  School type matched: 570,528 / 612,004 students
  Sampled: 160,000 rows, 80 countries
  Saved: pisa_student_2018_stratified.csv

Processing 2022...
  Students loaded: 613,744 rows
  School types loaded for 2022: 21,629 schools
  School type matched: 568,288 / 613,744 students
  Skipped 1 countries: ['CRI (too few students)']
  Sampled: 158,000 rows, 79 countries
  Saved: pisa_student_2022_stratified.csv

SAV years complete


In [37]:
# ── CELL 5: Process TXT years (2009, 2012) ────────────────────────────────────

txt_student_configs = {
    '2009': {
        'data':   '2009INT_STUDENT09_DEC11.txt',
        'syntax': 'PISA2009_SPSS_student.txt',
        'cols':   ['CNT', 'SCHOOLID', 'StIDStd', 'ESCS',
                   'PV1MATH', 'PV1READ', 'PV1SCIE',
                   'ST04Q01', 'W_FSTUWT'],
        'rename': {'StIDStd': 'STUDENTID', 'ST04Q01': 'GENDER',
                   'W_FSTUWT': 'STU_WEIGHT',
                   'PV1MATH': 'MATH', 'PV1READ': 'READ',
                   'PV1SCIE': 'SCIENCE'}
    },
    '2012': {
        'data':   '2012INT_STU12_DEC03.txt',
        'syntax': 'SPSS 2012 syntax to read in student questionnaire data file.txt',
        'cols':   ['CNT', 'SCHOOLID', 'StIDStd', 'ESCS',
                   'PV1MATH', 'PV1READ', 'PV1SCIE',
                   'ST04Q01', 'W_FSTUWT'],
        'rename': {'StIDStd': 'STUDENTID', 'ST04Q01': 'GENDER',
                   'W_FSTUWT': 'STU_WEIGHT',
                   'PV1MATH': 'MATH', 'PV1READ': 'READ',
                   'PV1SCIE': 'SCIENCE'}
    }
}

for year, config in txt_student_configs.items():
    print(f"\nProcessing {year}...")

    col_names, col_specs = parse_spss_syntax(sav_base + config['syntax'])

    # Filter to needed columns
    avail = [(n, s) for n, s in zip(col_names, col_specs)
             if n in config['cols']]
    av_names = [a[0] for a in avail]
    av_specs = [a[1] for a in avail]

    # Read in chunks
    chunks = []
    for i, chunk in enumerate(pd.read_fwf(
        sav_base + config['data'],
        colspecs=av_specs,
        names=av_names,
        encoding='latin-1',
        chunksize=10000
    )):
        chunks.append(chunk)
        if i % 10 == 0:
            print(f"  Processed {i*10000:,} rows...")

    df_stu = pd.concat(chunks, ignore_index=True)
    print(f"  Students loaded: {len(df_stu):,} rows")

    # Rename
    df_stu = df_stu.rename(columns=config['rename'])

    # Load and join school type
    df_sch = load_school_schtype(year, sav_base)
    df_stu = df_stu.merge(df_sch, on=['CNT', 'SCHOOLID'], how='left')
    matched = df_stu['SCHTYPE'].notna().sum()
    print(f"  School type matched: {matched:,} / {len(df_stu):,} students")

    # Stratified sample
    df_sampled = stratified_sample_pisa(df_stu, n_total=2000)
    df_sampled['YEAR'] = year
    print(f"  Sampled: {len(df_sampled):,} rows, "
          f"{df_sampled['CNT'].nunique()} countries")

    df_sampled.to_csv(
        processed_base + f'pisa_student_{year}_stratified.csv',
        index=False
    )
    print(f"  Saved: pisa_student_{year}_stratified.csv")
    all_years.append(df_sampled)

print("\nTXT years complete")


Processing 2009...
  Processed 0 rows...
  Processed 100,000 rows...
  Processed 200,000 rows...
  Processed 300,000 rows...
  Processed 400,000 rows...
  Processed 500,000 rows...
  Students loaded: 515,958 rows
  Columns found in 2009 school file: ['CNT', 'SCHOOLID', 'SCHTYPE']
  School file: processed 0 rows...
  School types loaded for 2009: 18,641 schools
  School type matched: 515,958 / 515,958 students
  Sampled: 146,658 rows, 74 countries
  Saved: pisa_student_2009_stratified.csv

Processing 2012...
  Processed 0 rows...
  Processed 100,000 rows...
  Processed 200,000 rows...
  Processed 300,000 rows...
  Processed 400,000 rows...
  Students loaded: 480,174 rows
  Columns found in 2012 school file: ['CNT', 'SCHOOLID', 'SCHLTYPE']
  School file: processed 0 rows...
  School types loaded for 2012: 18,139 schools
  School type matched: 480,174 / 480,174 students
  Skipped 1 countries: ['ALB (ESCS quartile error)']
  Sampled: 126,586 rows, 64 countries
  Saved: pisa_student_2012_s

In [14]:
# Check 2009 school column names
col_names_sch, _ = parse_spss_syntax(sav_base + 'PISA2009_SPSS_school.txt')
print("2009 school columns containing 'TYPE' or 'SCH':")
print([c for c in col_names_sch if 'TYPE' in c.upper() or 'SCHTYPE' in c.upper() or 'PUBLIC' in c.upper()])
print("\nFirst 50 columns:")
print(col_names_sch[:50])

2009 school columns containing 'TYPE' or 'SCH':
['SCHTYPE']

First 50 columns:
['CNT', 'COUNTRY', 'OECD', 'SUBNATIO', 'SCHOOLID', 'SC01Q01', 'SC01Q02', 'SC01Q03', 'SC01Q04', 'SC01Q05', 'SC01Q06', 'SC01Q07', 'SC01Q08', 'SC01Q09', 'SC01Q10', 'SC01Q11', 'SC01Q12', 'SC01Q13', 'SC01Q14', 'SC02Q01', 'SC03Q01', 'SC03Q02', 'SC03Q03', 'SC03Q04', 'SC04Q01', 'SC05Q01', 'SC06Q01', 'SC06Q02', 'SC07Q01', 'SC07Q02', 'SC08Q01', 'SC09Q11', 'SC09Q12', 'SC09Q21', 'SC09Q22', 'SC09Q31', 'SC09Q32', 'SC10Q01', 'SC10Q02', 'SC10Q03', 'SC11Q01', 'SC11Q02', 'SC11Q03', 'SC11Q04', 'SC11Q05', 'SC11Q06', 'SC11Q07', 'SC11Q08', 'SC11Q09', 'SC11Q10']


In [15]:
# Check if SCHTYPE is being parsed with correct position
col_names_sch, col_specs_sch = parse_spss_syntax(
    sav_base + 'PISA2009_SPSS_school.txt'
)

# Find SCHTYPE position
for name, spec in zip(col_names_sch, col_specs_sch):
    if name == 'SCHTYPE':
        print(f"SCHTYPE found at position: {spec}")

# Test reading just CNT, SCHOOLID, SCHTYPE
needed = ['CNT', 'SCHOOLID', 'SCHTYPE']
avail = [(n, s) for n, s in zip(col_names_sch, col_specs_sch)
         if n in needed]
av_names = [a[0] for a in avail]
av_specs = [a[1] for a in avail]

print(f"\nColumns found: {av_names}")

# Read 5 rows
df_test = pd.read_fwf(
    sav_base + '2009INT_SCHOOL09_Dec11.txt',
    colspecs=av_specs,
    names=av_names,
    encoding='latin-1',
    nrows=5
)
print(f"\nTest read:")
print(df_test.to_string())

SCHTYPE found at position: (367, 368)

Columns found: ['CNT', 'SCHOOLID', 'SCHTYPE']

Test read:
   CNT  SCHOOLID  SCHTYPE
0  ALB         1        1
1  ALB         2        1
2  ALB         3        1
3  ALB         4        1
4  ALB         5        1


In [18]:
col_names_2012_sch, _ = parse_spss_syntax(
    sav_base + 'SPSS 2012 syntax to read in school questionnaire data file.txt'
)
print("2012 school columns containing TYPE or PUBLIC or SCHLTYPE:")
print([c for c in col_names_2012_sch if any(x in c.upper() for x in
      ['TYPE', 'PUBLIC', 'SCHLTYPE', 'SCHTYPE', 'PRVT'])])

2012 school columns containing TYPE or PUBLIC or SCHLTYPE:
['SCHLTYPE']


In [38]:
# ── CELL 6: Combine, validate and save ────────────────────────────────────────

df_combined = pd.concat(all_years, ignore_index=True)

# Standardise columns
keep = ['YEAR', 'CNT', 'SCHOOLID', 'STUDENTID', 'GENDER',
        'ESCS', 'MATH', 'READ', 'SCIENCE', 'STU_WEIGHT', 'SCHTYPE']
df_combined = df_combined[[c for c in keep if c in df_combined.columns]]

print("=" * 60)
print("COMBINED STRATIFIED DATASET SUMMARY")
print("=" * 60)
print(f"Total rows:  {len(df_combined):,}")
print(f"Countries:   {df_combined['CNT'].nunique()}")
print(f"Years:       {sorted(df_combined['YEAR'].unique())}")

print("\nRows per year:")
print(df_combined.groupby('YEAR').size().to_string())

print("\nESCS distribution — students per quartile globally:")
df_combined['ESCS_Q'] = pd.qcut(
    df_combined['ESCS'].dropna(), q=4,
    labels=['Q1_low', 'Q2_mid', 'Q3_high', 'Q4_top']
)
print(df_combined['ESCS_Q'].value_counts().sort_index().to_string())
df_combined = df_combined.drop(columns=['ESCS_Q'])

print("\nSchool type distribution:")
print(df_combined['SCHTYPE'].value_counts().to_string())

print("\nGap stability check vs previous sample:")
print(f"{'CNT':<6} {'YEAR':<6} {'New gap':>10} {'Old gap':>10}")
old_gaps = {
    ('GBR','2022'): 70.32, ('EST','2022'): 70.51,
    ('KOR','2022'): 143.27, ('MAC','2022'): 29.53,
    ('FIN','2022'): 84.18
}
for (cnt, year), old_gap in old_gaps.items():
    group = df_combined[
        (df_combined['CNT'] == cnt) &
        (df_combined['YEAR'] == year)
    ].dropna(subset=['ESCS', 'MATH'])
    if len(group) == 0:
        continue
    q25 = group['ESCS'].quantile(0.25)
    q75 = group['ESCS'].quantile(0.75)
    bottom = group[group['ESCS'] <= q25]['MATH'].mean()
    top    = group[group['ESCS'] >= q75]['MATH'].mean()
    new_gap = round(top - bottom, 2)
    print(f"{cnt:<6} {year:<6} {new_gap:>10.2f} {old_gap:>10.2f}")

    # Update margin of error display in Cell 6
n_per_q = 500
se = 100 / np.sqrt(n_per_q)
gap_margin = 1.96 * np.sqrt(2) * se
print(f"\nStatistical note:")
print(f"Sample: 2000 students per country (500 per ESCS quartile)")
print(f"Margin of error on gap estimates: ±{gap_margin:.1f} pts (95% CI)")
print(f"Relative error: {gap_margin/80*100:.1f}%")
print(f"Minimum detectable difference: {gap_margin*np.sqrt(2):.1f} pts")

# Save — overwrites previous version
output_path = processed_base + 'pisa_student_all_years.csv'
df_combined.to_csv(output_path, index=False)
print(f"\nSaved: pisa_student_all_years.csv")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")
print("\nData preparation complete — ready to rerun EDA and model notebooks")

COMBINED STRATIFIED DATASET SUMMARY
Total rows:  734,122
Countries:   107
Years:       ['2009', '2012', '2015', '2018', '2022']

Rows per year:
YEAR
2009    146658
2012    126586
2015    142878
2018    160000
2022    158000

ESCS distribution — students per quartile globally:
ESCS_Q
Q1_low     183534
Q2_mid     183536
Q3_high    183736
Q4_top     183316

School type distribution:
SCHTYPE
3.0    392428
1.0    180749
2.0     95775
9.0     17260
7.0     12128
8.0        34

Gap stability check vs previous sample:
CNT    YEAR      New gap    Old gap
GBR    2022        94.72      70.32
EST    2022        74.58      70.51
KOR    2022       107.52     143.27
MAC    2022        50.55      29.53
FIN    2022        95.35      84.18

Statistical note:
Sample: 2000 students per country (500 per ESCS quartile)
Margin of error on gap estimates: ±12.4 pts (95% CI)
Relative error: 15.5%
Minimum detectable difference: 17.5 pts

Saved: pisa_student_all_years.csv
File size: 58.9 MB

Data preparation comp

In [30]:
import scipy.stats as stats

# For a typical maths score std dev of ~100 points
# How large is our margin of error with 75 students per quartile?

std_dev = 100  # PISA maths standard deviation
n = 75         # students per quartile

# Standard error of the mean
se = std_dev / np.sqrt(n)

# 95% confidence interval margin
margin_95 = 1.96 * se

# For gap calculation we compare two groups
# so the margin doubles
gap_margin = 1.96 * np.sqrt(2) * se

print(f"Per quartile (n={n}):")
print(f"  Standard error of mean:     {se:.1f} pts")
print(f"  95% CI margin (single mean): ±{margin_95:.1f} pts")
print(f"  95% CI margin (gap estimate): ±{gap_margin:.1f} pts")
print(f"\nInterpretation:")
print(f"  Our average gap is ~80 pts")
print(f"  Margin of error on gap: ±{gap_margin:.1f} pts")
print(f"  Relative error: {gap_margin/80*100:.1f}%")

# Minimum detectable difference between two countries
mdd = gap_margin * np.sqrt(2)
print(f"\nMinimum detectable difference between two countries: {mdd:.1f} pts")
print(f"(i.e. gaps must differ by >{mdd:.0f} pts to be statistically distinguishable)")

Per quartile (n=75):
  Standard error of mean:     11.5 pts
  95% CI margin (single mean): ±22.6 pts
  95% CI margin (gap estimate): ±32.0 pts

Interpretation:
  Our average gap is ~80 pts
  Margin of error on gap: ±32.0 pts
  Relative error: 40.0%

Minimum detectable difference between two countries: 45.3 pts
(i.e. gaps must differ by >45 pts to be statistically distinguishable)


In [31]:
import scipy.stats as stats

for n in [75, 125, 167, 250]:
    std_dev = 100
    se = std_dev / np.sqrt(n)
    gap_margin = 1.96 * np.sqrt(2) * se
    total = n * 4 * 107 * 5
    print(f"n={n:3d} per quartile ({n*4:3d}/country): "
          f"gap margin ±{gap_margin:.1f} pts | "
          f"relative error {gap_margin/80*100:.1f}% | "
          f"total rows {total:,}")

n= 75 per quartile (300/country): gap margin ±32.0 pts | relative error 40.0% | total rows 160,500
n=125 per quartile (500/country): gap margin ±24.8 pts | relative error 31.0% | total rows 267,500
n=167 per quartile (668/country): gap margin ±21.4 pts | relative error 26.8% | total rows 357,380
n=250 per quartile (1000/country): gap margin ±17.5 pts | relative error 21.9% | total rows 535,000


In [32]:
for n in [75, 125, 250, 500, 1500, 6000]:
    std_dev = 100
    se = std_dev / np.sqrt(n)
    gap_margin = 1.96 * np.sqrt(2) * se
    print(f"n={n:5d} per quartile: gap margin ±{gap_margin:.1f} pts | "
          f"relative error {gap_margin/80*100:.1f}%")

n=   75 per quartile: gap margin ±32.0 pts | relative error 40.0%
n=  125 per quartile: gap margin ±24.8 pts | relative error 31.0%
n=  250 per quartile: gap margin ±17.5 pts | relative error 21.9%
n=  500 per quartile: gap margin ±12.4 pts | relative error 15.5%
n= 1500 per quartile: gap margin ±7.2 pts | relative error 8.9%
n= 6000 per quartile: gap margin ±3.6 pts | relative error 4.5%


In [33]:
# Check minimum students per country in full 2022 dataset
print("Countries with fewer than 2000 students in 2022 full dataset:")
small = df_2022_full.dropna(subset=['ESCS']).groupby('CNT').size()
print(small[small < 2000].to_string())
print(f"\nTotal countries: {len(small)}")
print(f"Countries with 2000+ students: {(small >= 2000).sum()}")
print(f"Countries with 500+ per quartile (2000 total): {(small >= 2000).sum()}")

Countries with fewer than 2000 students in 2022 full dataset:
Series([], )

Total countries: 79
Countries with 2000+ students: 79
Countries with 500+ per quartile (2000 total): 79


In [34]:
# Check all years have enough students for 2000 per country
for year in ['2009', '2012', '2015', '2018']:
    fname_map = {
        '2015': '2015CY6_MS_CMB_STU_QQQ.sav',
        '2018': '2018CY07_MSU_STUDENT_QQQ.sav'
    }

    if year in ['2015', '2018']:
        df_test, _ = pyreadstat.read_sav(
            sav_base + fname_map[year],
            usecols=['CNT', 'ESCS']
        )
    else:
        # Use already loaded full CSVs from archive
        df_test = pd.read_csv(
            base + f'archive/pisa_student_{year}_full.csv',
            usecols=['CNT', 'ESCS']
        )

    counts = df_test.dropna(subset=['ESCS']).groupby('CNT').size()
    under_2000 = counts[counts < 2000]
    print(f"{year}: {len(counts)} countries | "
          f"Under 2000 students: {len(under_2000)} | "
          f"Min: {counts.min():,} | Mean: {counts.mean():.0f}")

2009: 74 countries | Under 2000 students: 2 | Min: 329 | Mean: 6972
2012: 65 countries | Under 2000 students: 2 | Min: 293 | Mean: 7387
2015: 72 countries | Under 2000 students: 4 | Min: 1,385 | Mean: 6991
2018: 80 countries | Under 2000 students: 1 | Min: 1,995 | Mean: 7470
